Compare raw HVGs vs. the State Embedding on a Tahoe cell line.

Input: `state_infer/c36.h5ad` (heldout Tahoe cell line), which contains:
- Raw gene expression in `adata.X` and the 2000 Tahoe HVGs in `adata.obsm['X_hvg']`
- The SE-600M State Embedding in `adata.obsm['X_state']` (2058-dim)
- Perturbation metadata in `.obs`: `drugname_drugconc`, `plate`, `cell_name`, with `DMSO_TF` controls

We run ST inference two ways and compare:
- `X_hvg`  → **ST-HVG-Tahoe** checkpoint
- `X_state` → **ST-SE-Tahoe** checkpoint

In [7]:
! pip -q install matplotlib-inline scanpy

import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import os

# backed read — c36 is 39k cells x 62710 genes; keep .X on disk, obsm is small
adata = ad.read_h5ad("state_infer/c36.h5ad", backed="r")
print(adata.shape, {k: v.shape for k, v in adata.obsm.items()})

(39463, 62710) {'X_hvg': (39463, 2000), 'X_state': (39463, 2058)}


## Embed with the State Embedding (SE-600M) model
c36 already ships with `obsm['X_state']`. If you want to regenerate it with your own SE script, run it below and write the result back into `adata.obsm['X_state']` (2058-dim) so the `X_state` inference below picks it up.

In [8]:
! pip -q install -U "huggingface_hub>=0.22.0"
from huggingface_hub import snapshot_download

repo_id = "arcinstitute/SE-600M"
target_dir = "SE-600M"
exclude = ["se600m_epoch4.ckpt", "se600m_epoch4.safetensors"]

if os.path.exists(target_dir) and os.listdir(target_dir):
    raise SystemExit(f"'{target_dir}' already exists and is not empty.")

snapshot_download(
    repo_id=repo_id,
    repo_type="model",
    revision="main",
    local_dir=target_dir,
    local_dir_use_symlinks=False,  # copy files instead of symlinking from cache
    ignore_patterns=exclude,       # skip the big epoch-4 files
    resume_download=True,
)

print(f"Downloaded to ./{target_dir} (excluded: {', '.join(exclude)})")

/home/alaysia/miniconda/envs/state/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/home/alaysia/miniconda/envs/state/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded to ./SE-600M (excluded: se600m_epoch4.ckpt, se600m_epoch4.safetensors)


In [11]:
# Embed c36 with SE-600M. Run in-process (not the `state` CLI) so we can opt in to
# writing nullable-string obs columns — c36.obs has some (e.g. BARCODE_SUB_LIB_ID),
# and anndata blocks writing them by default. The setting can't cross into the
# CLI subprocess, so the `! state emb transform` version fails at write time.
import anndata
import torch
from state.emb.inference import Inference

anndata.settings.allow_write_nullable_strings = True

pe = torch.load("SE-600M/protein_embeddings.pt", weights_only=False, map_location="cpu")
inferer = Inference(cfg=None, protein_embeds=pe)          # cfg=None -> use config baked into ckpt
inferer.load_model("SE-600M/se600m_epoch16.ckpt")

inferer.encode_adata(
    input_adata_path="state_infer/c36.h5ad",
    output_adata_path="state_infer/c36_emb.h5ad",         # keeps .X, .obs, existing obsm; adds X_state
    emb_key="X_state",
)

!!! 19648 genes mapped to embedding file (out of 62710)


Encoding: 100%|██████████| 4933/4933 [04:24<00:00, 18.66it/s]


array([[ 0.01286538, -0.01215064, -0.05495582, ..., -0.12792969,
         0.14550781, -0.23828125],
       [ 0.01816726,  0.01793335,  0.00354768, ...,  0.01611328,
         0.21386719, -0.20117188],
       [ 0.0280238 , -0.01897611, -0.04419753, ..., -0.03320312,
         0.3125    ,  0.06005859],
       ...,
       [-0.01580784,  0.01476995, -0.00445094, ..., -0.0112915 ,
         0.19335938, -0.09472656],
       [ 0.02816681, -0.03664886, -0.01072259, ..., -0.1328125 ,
         0.26171875, -0.25390625],
       [-0.01562711, -0.0098571 , -0.03862702, ...,  0.04882812,
         0.296875  ,  0.07714844]], shape=(39463, 2058), dtype=float32)

In [ ]:
# Sanitize c36_emb.h5ad for the `state tx infer` subprocess, which can't toggle
# allow_write_nullable_strings. The encode step writes some string fields as
# nullable-string-array (here the obs index BARCODE_SUB_LIB_ID and the var index
# gene_name). anndata reads those back as pd StringArray and then refuses to
# rewrite them, so the CLI dies at its final adata.write_h5ad(). A string-array
# doesn't help (it round-trips back to StringArray); categorical does. We convert
# every nullable-string-array element to categorical in place (leaves .X untouched).
import h5py, pandas as pd
from anndata._io.specs import read_elem, write_elem

f = "state_infer/c36_emb.h5ad"
hits = []
with h5py.File(f, "r") as h:
    h.visititems(lambda n, o: hits.append(n)
                 if o.attrs.get("encoding-type") == "nullable-string-array" else None)

with h5py.File(f, "r+") as h:
    for name in hits:
        parent, key = h[name].parent, name.rsplit("/", 1)[-1]
        cat = pd.Categorical(pd.Series(read_elem(h[name])).fillna("").astype(str))
        del h[name]
        write_elem(parent, key, cat)
        print(f"sanitized {name} -> categorical ({len(cat.categories)} cats)")
print("nullable-string fields fixed:", len(hits))

## Run Inference
Here we run inference on the HVG/State embedded data so that we can compare the clustering before and after inference.


This is how the data will look after running these inference cells:
| Representation | File | Key | Dim |
| -------- | -------- | -------- | -------- |
| HVG Before  | `c36_emb.h5ad`  | `obsm['X_hvg']`  | 2000  |
| HVG After  | `c36_hvg_simulated.h5ad`  | `obsm['X_hvg']`  | 2000  |
| State Before  | `c36_emb.h5ad`  | `obsm['X_state']`  | 2058  |
| State After  | `c36_emb_simulated.h5ad`  | `obsm['X_state']`  | 2058  |

### ...on X_hvg (raw HVGs, ST-HVG-Tahoe)

In [ ]:
# Download the HVG checkpoint (ST-HVG-Tahoe) — the raw-HVG counterpart to
# ST-SE-Tahoe. Uses embed_key=X_hvg (2000-dim Tahoe HVGs). Grabs the checkpoint
# plus the config, one-hot maps, and var_dims that the `state tx infer` CLI loads.
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="arcinstitute/ST-HVG-Tahoe",
    allow_patterns=[
        "fewshot/state_generalization_X_hvg/checkpoints/best.ckpt",
        "fewshot/state_generalization_X_hvg/config.yaml",
        "fewshot/state_generalization_X_hvg/*.pkl",   # batch/cell_type onehot, var_dims
        "fewshot/state_generalization_X_hvg/*.pt",    # pert_onehot_map
        "fewshot/state_generalization_X_hvg/*.yaml",
        "fewshot/state_generalization_X_hvg/*.txt",
        "fewshot/state_generalization_X_hvg/*.json",
    ],
    local_dir="state_infer/ST-HVG-Tahoe",
)

In [ ]:
# HVG inference with ST-HVG-Tahoe. Input c36_emb.h5ad already carries obsm['X_hvg']
# (2000-d observed HVGs, identical to c36.h5ad) and is sanitized (obs/var indices are
# categorical, not nullable strings), so the CLI can write its output. Predictions
# land in the output's obsm['X_hvg']; obsm['X_state'] passes through unchanged.
! state tx infer \
    --model-dir state_infer/ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/ \
    --checkpoint state_infer/ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/checkpoints/best.ckpt \
    --pert-col drugname_drugconc \
    --batch-col plate \
    --control-pert "[('DMSO_TF', 0.0, 'uM')]" \
    --embed-key "X_hvg" \
    --adata state_infer/c36_emb.h5ad \
    --output state_infer/c36_hvg_simulated.h5ad

### ...on X_state (SE-600M embedding, ST-SE-Tahoe)
Requires `obsm['X_state']` in the input file. c36 already ships with it; if you re-embed with your own SE script, write the result into `obsm['X_state']` first.

In [5]:
# Download the State-Embedding checkpoint (ST-SE-Tahoe) — the SE counterpart to
# ST-HVG-Tahoe. Uses embed_key=X_state (2058-dim SE-600M embedding).
%pip install -q --upgrade huggingface_hub

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="arcinstitute/ST-SE-Tahoe",
    # allow_patterns="fewshot/state_generalization_X_state/*",
    allow_patterns=[
        "fewshot/state_generalization_X_state/checkpoints/best.ckpt",
        "fewshot/state_generalization_X_state/*.json",
        "fewshot/state_generalization_X_state/*.yaml",
        "fewshot/state_generalization_X_state/*.yml",
        "fewshot/state_generalization_X_state/*.txt",
        "fewshot/state_generalization_X_state/configs/*",
        "fewshot/state_generalization_X_state/hparams.yaml",
    ],
    local_dir="state_infer/ST-SE-Tahoe",
)

Note: you may need to restart the kernel to use updated packages.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

'/home/alaysia/relearn/notebooks/alaysia/state_infer/ST-SE-Tahoe'

In [9]:
# NOTE: X_state needs the ST-SE-Tahoe repo (NOT ST-HVG-Tahoe), with the state_infer/ prefix.
! state tx infer \
    --model-dir state_infer/ST-SE-Tahoe/fewshot/state_generalization_X_state/ \
    --checkpoint state_infer/ST-SE-Tahoe/fewshot/state_generalization_X_state/checkpoints/best.ckpt \
    --pert-col drugname_drugconc \
    --batch-col plate \
    --control-pert "[('DMSO_TF', 0.0, 'uM')]" \
    --embed-key "X_state" \
    --adata state_infer/c36_emb.h5ad \
    --output state_infer/c36_emb_simulated.h5ad

==> STATE: tx infer (virtual experiment)
Loaded config: state_infer/ST-SE-Tahoe/fewshot/state_generalization_X_state/config.yaml
Control perturbation: [('DMSO_TF', 0.0, 'uM')]
Grouping by cell type column: cell_name
Loaded batch one-hot map from: state_infer/ST-SE-Tahoe/fewshot/state_generalization_X_state/batch_onehot_map.pkl
Loaded cell type one-hot map from: state_infer/ST-SE-Tahoe/fewshot/state_generalization_X_state/cell_type_onehot_map.pkl
StateTransitionPerturbationModel(
  (loss_fn): SamplesLoss()
  (gene_decoder): LatentToGeneDecoder(
    (decoder): Sequential(
      (0): Linear(in_features=2058, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=1024, out_features=1024, bias=True)
      (5): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (6): GELU(approximate='none')
      (7): Dropout(p=0.1, inplace=False)
     

# Analysis
To compare how each of the two representations perform before versus after inference by various calculation methods.

## kNN Label Recoverability

## Silhouette Score and Average Silhouette Width (ASW)

## Adjusted Rand Index (ARI) and Normalized Mutual Information (NMI)
Between Leiden clusters and labels